In [ ]:
%%writefile myinfer.py

from warnings import filterwarnings
filterwarnings("ignore")

import os
import glob
import timm
import torch
import torch.nn as nn
import torchaudio
import torchvision
import numpy as np
import pandas as pd
import openvino as ov
from tqdm import tqdm
from torch.utils.data import Dataset

# ─── Configuration ────────────────────────────────────────────────────────────

ov_device       = "CPU"

PATH            = '/kaggle/input/competitions/birdclef-2026/'
TEST_PATH       = PATH + 'test_soundscapes/'
TRAIN_PATH      = PATH + 'train_audio/' # Corrected fallback path
SUBMISSION_FILE = "submission.csv"
model_dir       = "/kaggle/input/notebooks/antoinemasq/birdclef-2026-pytorch-baseline-training/models"

DUR = 5
SR  = 32000

# ─── Spectrogram ─────────────────────────────────────────────────────────────

class Spectrogram(nn.Module):
    def __init__(self, sr=32000, n_fft=2048, n_mels=256, hop_length=512,
                 f_min=20, f_max=16000, channels=1, norm="slaney",
                 mel_scale="htk", target_size=(256, 256), top_db=80.0, **kwargs):
        super().__init__()
        self.channels = channels
        self.top_db   = top_db
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr, n_fft=n_fft, hop_length=hop_length,
            n_mels=n_mels, f_min=f_min, f_max=f_max,
            mel_scale=mel_scale, pad_mode="reflect", power=2.0,
            norm=norm, center=True,
        )
        self.resize = torchvision.transforms.Resize(size=target_size)

    def power_to_db(self, S):
        amin     = 1e-10
        log_spec = 10.0 * torch.log10(S.clamp(min=amin))
        log_spec -= 10.0 * torch.log10(torch.tensor(amin).to(S))
        if self.top_db is not None:
            max_val  = log_spec.flatten(-2).max(dim=-1).values[..., None, None]
            log_spec = torch.maximum(log_spec, max_val - self.top_db)
        return log_spec

    def forward(self, x, resize=True):
        squeeze = x.dim() == 1
        if squeeze:
            x = x.unsqueeze(0)
        mel = self.mel_transform(x)
        mel = self.power_to_db(mel)
        mel = mel.unsqueeze(1).repeat(1, self.channels, 1, 1)
        if resize:
            mel = self.resize(mel)
        B, C = mel.shape[:2]
        flat = mel.view(B, C, -1)
        mins = flat.min(dim=-1).values[..., None, None]
        maxs = flat.max(dim=-1).values[..., None, None]
        mel  = (mel - mins) / (maxs - mins + 1e-7)
        if squeeze:
            mel = mel.squeeze(0)
        return mel

# ─── Model ───────────────────────────────────────────────────────────────────

class BirdModel(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        cfg = {
            'backbone':        'tf_efficientnetv2_b0',
            'backbone_pooling':'avg',
            'dropout':          0.1,
            'pretrained':       False,
            'channels':         1,
            'num_labels':       234, # Will dynamically update
        }
        if config:
            cfg.update(config)
        self.backbone = timm.create_model(
            cfg['backbone'],
            pretrained=cfg['pretrained'],
            num_classes=cfg['num_labels'],
            global_pool=cfg['backbone_pooling'],
            in_chans=cfg['channels'],
            drop_rate=cfg['dropout'],
        )

    def forward(self, x):
        return self.backbone(x)

# ─── Dataset ─────────────────────────────────────────────────────────────────
# ─── Upgraded Dataset with Overlapping TTA ────────────────────────────────────

class BirdDataset(Dataset):
    def __init__(self, paths, spec_transform):
        self.paths = paths
        self.spec  = spec_transform
        self.chunk_len = SR * DUR
        self.half_chunk = self.chunk_len // 2

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        filepath = self.paths[idx]
        filename = filepath.split('/')[-1].split('.')[0]
        try:
            wav, _ = torchaudio.load(filepath)
            wav = wav.float()[0]  
            
            n_seg = len(wav) // self.chunk_len 
            
            if n_seg == 0:
                wav = torch.nn.functional.pad(wav, (0, self.chunk_len - len(wav)))
                n_seg = 1
            else:
                wav = wav[: n_seg * self.chunk_len]
            
            # 1. Standard Chunks (0-5s, 5-10s, ...)
            wav_std = wav.reshape((n_seg, self.chunk_len))
            
            # 2. Shifted Chunks for TTA (2.5-7.5s, 7.5-12.5s, ...)
            wav_padded = torch.nn.functional.pad(wav, (0, self.half_chunk))
            wav_shift = torch.stack([
                wav_padded[i * self.chunk_len + self.half_chunk : (i+1) * self.chunk_len + self.half_chunk]
                for i in range(n_seg)
            ])
            
            mel_std   = torch.stack([self.spec(wav_std[i]) for i in range(n_seg)])
            mel_shift = torch.stack([self.spec(wav_shift[i]) for i in range(n_seg)])
            names =[f"{filename}_{int((i + 1) * DUR)}" for i in range(n_seg)]
            
            return mel_std.numpy().astype(np.float32), mel_shift.numpy().astype(np.float32), names
            
        except Exception as e:
            print(f"Error loading {filepath}: {e}")
            n_seg = int(240 / DUR) 
            mel_std   = torch.zeros((n_seg, 1, 256, 256))
            mel_shift = torch.zeros((n_seg, 1, 256, 256))
            names =[f"{filename}_{int((i + 1) * DUR)}" for i in range(n_seg)]
            return mel_std.numpy().astype(np.float32), mel_shift.numpy().astype(np.float32), names

# ─── OpenVINO Export & Compile ───────────────────────────────────────────────

def export_to_openvino(ckpt_path, ov_model_path, config=None):
    if os.path.exists(ov_model_path):
        return
    state = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    
    num_labels = state['backbone.classifier.weight'].shape[0]
    cfg = config.copy() if config else {}
    cfg['num_labels'] = num_labels
    
    model = BirdModel(cfg)
    model.load_state_dict(state)
    model.eval()
    
    dummy    = torch.zeros(1, 1, 256, 256)
    ov_model = ov.convert_model(model, example_input=dummy)
    ov_model.reshape([-1, 1, 256, 256]) # Support dynamic batch sizes
    ov.save_model(ov_model, ov_model_path)

def compile_ov_model(ov_model_path):
    core     = ov.Core()
    ov_model = core.read_model(ov_model_path)
    return core.compile_model(ov_model, device_name=ov_device)

# ─── Aggressive Inference & Post-Processing ───────────────────────────────────
# ─── Safe, Non-Clipping Inference ─────────────────────────────────────────────

def predict_batch(compiled_models, batch_np):
    logits_list =[]
    for compiled in compiled_models:
        logits = compiled(batch_np)[compiled.output(0)]
        logits_list.append(logits)
    
    # 1. Ensemble average the logits directly
    mean_logits = np.mean(logits_list, axis=0)
    
    # 2. Standard sigmoid (removed temperature scaling to preserve native calibration)
    probs = 1.0 / (1.0 + np.exp(np.clip(-mean_logits, -50, 50)))
    return probs

def run_inference(compiled_models, paths, spec):
    dataset  = BirdDataset(paths, spec)
    all_preds, all_names = [],[]

    for mel_std, mel_shift, names in tqdm(dataset, desc="Infer"):
        
        preds_std   = predict_batch(compiled_models, mel_std)
        preds_shift = predict_batch(compiled_models, mel_shift)
        
        # 1. Weave TTA Shifted Predictions (Recovers birds cut in half)
        preds = np.copy(preds_std)
        n = len(preds)
        for i in range(n):
            shift_prev = preds_shift[i-1] if i > 0 else preds_std[i]
            shift_curr = preds_shift[i]
            # 50% Standard Window + 50% overlapping halves
            preds[i] = 0.50 * preds_std[i] + 0.25 * shift_prev + 0.25 * shift_curr
        
        # 2. Wider Gaussian Time-Domain Smoothing
        smoothed_preds = np.copy(preds)
        for i in range(n):
            p_m2 = preds[max(0, i-2)]
            p_m1 = preds[max(0, i-1)]
            p_0  = preds[i]
            p_p1 = preds[min(n-1, i+1)]
            p_p2 = preds[min(n-1, i+2)]
            # Smooth the probabilities out across a 25-second window
            smoothed_preds[i] = 0.05*p_m2 + 0.15*p_m1 + 0.60*p_0 + 0.15*p_p1 + 0.05*p_p2
            
        # 3. Global File Context (The Grandmaster Trick, without clipping!)
        file_max = np.max(smoothed_preds, axis=0)
        
        # Blend: 80% local chunk + 20% global file context
        # This securely boosts the AUC rank of chunks in files where the bird is confirmed to exist
        final_preds = 0.80 * smoothed_preds + 0.20 * file_max
        
        all_preds.append(final_preds)
        all_names.extend(names)

    return np.concatenate(all_preds, axis=0), all_names
# ─── Submission ───────────────────────────────────────────────────────────────

def create_submission(preds, names, train_labels):
    sample_sub = pd.read_csv(PATH + 'sample_submission.csv')
    expected_cols = sample_sub.columns.tolist()
    
    # Check if predictions are completely empty
    if len(preds) == 0:
        print("No predictions generated. Saving sample_submission directly.")
        sample_sub.to_csv(SUBMISSION_FILE, index=False)
        return sample_sub
        
    df_preds = pd.DataFrame(preds, columns=train_labels[:preds.shape[1]])
    df_preds['row_id'] = names
    
    final_dict = {'row_id': df_preds['row_id']}
    for col in expected_cols:
        if col == 'row_id': continue
        if col in df_preds.columns:
            final_dict[col] = df_preds[col].values
        else:
            final_dict[col] = 0.0
            
    final_df = pd.DataFrame(final_dict)
    final_df = final_df[expected_cols]
    final_df.to_csv(SUBMISSION_FILE, index=False)
    
    print(f"---> Submission shape = {final_df.shape}")
    return final_df

# ─── Main ─────────────────────────────────────────────────────────────────────

def main():
    train_labels = sorted(pd.read_csv(PATH + 'train.csv')['primary_label'].unique().tolist())
    
    # 1. Search for test files (Using glob to support recursive search)
    paths = glob.glob(TEST_PATH + "**/*.ogg", recursive=True)
    
    # 2. Fallback for interactive mode
    if not paths:
        paths = glob.glob(TRAIN_PATH + "**/*.ogg", recursive=True)[:2]

    # 3. Handle absolutely no files case elegantly!
    if not paths:
        print(f"Files: 0. Generating default submission directly to avoid crashes.")
        create_submission([],[], train_labels)
        return

    print(f"Files: {len(paths)}, Model Output Classes: {len(train_labels)}")

    spec = Spectrogram(sr=SR, n_fft=2048, n_mels=256, hop_length=512,
                       f_min=20, f_max=16000, channels=1,
                       target_size=(256, 256), top_db=80.0)

    all_ckpts =[os.path.join(model_dir, f) for f in sorted(os.listdir(model_dir)) if f.endswith('.pth')]
    best_ckpts =[p for p in all_ckpts if '_best.pth' in os.path.basename(p)]
    ckpt_paths = best_ckpts if best_ckpts else all_ckpts

    ov_export_dir = "/kaggle/working/ov_models"
    os.makedirs(ov_export_dir, exist_ok=True)

    compiled_models =[]
    for ckpt in ckpt_paths:
        ov_name = os.path.splitext(os.path.basename(ckpt))[0] + '.xml'
        ov_path = os.path.join(ov_export_dir, ov_name)
        export_to_openvino(ckpt, ov_path)
        compiled_models.append(compile_ov_model(ov_path))

    print(f"Loaded {len(compiled_models)} OpenVINO model(s)")

    preds, names = run_inference(compiled_models, paths, spec)
    create_submission(preds, names, train_labels)

if __name__ == "__main__":
    main()

In [ ]:
import os, sys, time 
import pandas as pd

!python myinfer.py
print()

print("\n\n\n")
display(
    pd.read_csv("submission.csv", index_col = "row_id").
    head(5)
)